In [1]:
import os

import numpy as np
import pandas as pd
import xarray as xr
from bluemath_tk.core.io import load_model
from bluemath_tk.core.operations import get_degrees_from_uv, get_uv_components
from bluemath_tk.datamining.mda import MDA
from bluemath_tk.datamining.pca import PCA
from bluemath_tk.interpolation.gps import ExactGPInterpolation

In [2]:
num_mda_points = 500

In [3]:
mda_models = []
pca_models = []
gp_models = []

for goal in range(1, 11):
    print(f"Processing goal {goal}")
    metamodel_cases_goal = xr.open_dataset(
        f"SnapWave_CASES_METAMODEL/10mDepthAndBuoysPoints_{goal}.nc"
    )
    wavedir_u, wavedir_v = get_uv_components(metamodel_cases_goal.point_wavdir)
    metamodel_cases_goal["point_wavdir_u"] = wavedir_u
    metamodel_cases_goal["point_wavdir_v"] = wavedir_v
    metamodel_cases_goal["stations"] = range(177)
    metamodel_cases_df_for_goal = metamodel_cases_goal[
        ["tp", "wd", "ds", "zs"]
    ].to_dataframe()

    if os.path.exists(f"outputs/mda_{num_mda_points}/mda_model_{goal}.pkl"):
        mda = load_model(f"outputs/mda_{num_mda_points}/mda_model_{goal}.pkl")
    else:
        os.makedirs(f"outputs/mda_{num_mda_points}", exist_ok=True)
        mda = MDA(num_centers=num_mda_points)
        mda.fit(metamodel_cases_df_for_goal, directional_variables=["wd"])
        # mda.plot_selected_centroids();
        mda.save_model(f"outputs/mda_{num_mda_points}/mda_model_{goal}.pkl")
    mda_models.append(mda)

    if os.path.exists(f"outputs/pca_{num_mda_points}/pca_model_{goal}.pkl"):
        pca = load_model(f"outputs/pca_{num_mda_points}/pca_model_{goal}.pkl")
    else:
        os.makedirs(f"outputs/pca_{num_mda_points}", exist_ok=True)
        pca = PCA(n_components=0.95)
        pcs_ds = pca.fit_transform(
            data=metamodel_cases_goal.isel(case_num=mda.centroid_real_indices),
            vars_to_stack=["point_hm0", "point_tp", "point_wavdir_u", "point_wavdir_v"],
            coords_to_stack=["stations"],
            pca_dim_for_rows="case_num",
        )
        pca.save_model(f"outputs/pca_{num_mda_points}/pca_model_{goal}.pkl")
    pca_models.append(pca)

    if os.path.exists(f"outputs/gp_{num_mda_points}/gp_model_{goal}.pkl"):
        gp = load_model(f"outputs/gp_{num_mda_points}/gp_model_{goal}.pkl")
    else:
        os.makedirs(f"outputs/gp_{num_mda_points}", exist_ok=True)
        gp = ExactGPInterpolation()
        gp.fit(
            subset_data=mda.centroids[["tp", "wd", "ds", "zs"]],
            subset_directional_variables=["wd"],
            target_data=pca.pcs_df,
        )
        gp.save_model(f"outputs/gp_{num_mda_points}/gp_model_{goal}.pkl")
    gp_models.append(gp)

Processing goal 1
Processing goal 2
Processing goal 3
Processing goal 4
Processing goal 5
Processing goal 6
Processing goal 7
Processing goal 8
Processing goal 9
Processing goal 10


In [4]:
len(mda_models), len(pca_models), len(gp_models)

(10, 10, 10)

In [ ]:
reconstructed_months = []

for month, month_str in zip(
    ["2011-12"],
    ["201112_500"],
):
    # Set list to save the predictions
    reconstruction_for_each_goal = []
    # Load the reanalysis timeseries
    reanalysis_ds = xr.open_zarr("inputs/ERA5/processed/ERA5_netherlands.zarr").sel(
        time=month
    )

    for partition, variables in {
        # "bulk": ["hs", "fp", "dp", "spr"],
        # "sea": ["phs0", "ptp0", "pdp0", "pspr0"],
        # "swell1": ["phs1", "ptp1", "pdp1", "pspr1"],
        # "swell2": ["phs2", "ptp2", "pdp2", "pspr2"],
        # "swell3": ["phs3", "ptp3", "pdp3", "pspr3"],
        "bulk": ["swh", "pp1d", "mwp", "wdw"],
        "sea": ["shww", "mpww", "mdww", "dwww"],
        "swell1": ["swh1", "mwp1", "mwd1", "dwps"],
        "swell2": ["swh2", "mwp2", "mwd2", "dwps"],
        "swell3": ["swh3", "mwp3", "mwd3", "dwps"],
    }.items():
        # Iterate over all the goals
        for goal in range(1, 11):
            # First, set the proper variables
            if partition == "bulk":
                reanalysis_ds_for_goal = reanalysis_ds.sel(goal=goal)[variables].rename(
                    {
                        "swh": "hs",
                        "pp1d": "tp",
                        "mwp": "wd",
                        "wdw": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()
                # reanalysis_df_for_goal["tp"] = 1 / reanalysis_df_for_goal["tp"]
            elif partition == "sea":
                reanalysis_ds_for_goal = reanalysis_ds.sel(goal=goal)[variables].rename(
                    {
                        "shww": "hs",
                        "mpww": "tp",
                        "mdww": "wd",
                        "dwww": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()
            elif partition == "swell1":
                reanalysis_ds_for_goal = reanalysis_ds.sel(goal=goal)[variables].rename(
                    {
                        "swh1": "hs",
                        "mwp1": "tp",
                        "mwd1": "wd",
                        "dwps": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()
            elif partition == "swell2":
                reanalysis_ds_for_goal = reanalysis_ds.sel(goal=goal)[variables].rename(
                    {
                        "swh2": "hs",
                        "mwp2": "tp",
                        "mwd2": "wd",
                        "dwps": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()
            elif partition == "swell3":
                reanalysis_ds_for_goal = reanalysis_ds.sel(goal=goal)[variables].rename(
                    {
                        "swh3": "hs",
                        "mwp3": "tp",
                        "mwd3": "wd",
                        "dwps": "ds",
                    }
                )
                reanalysis_df_for_goal = reanalysis_ds_for_goal.to_dataframe()

            # Add water level
            reanalysis_df_for_goal["zs"] = (
                xr.open_zarr("inputs/CODEC/processed/CODEC_netherlands.zarr")
                .sel(time=month, goal=goal)
                .waterlevel.values
            )

            # Predict in buoys
            prediction_ds_in_points = pca_models[goal - 1].inverse_transform(
                xr.Dataset(
                    {
                        "PCs": (
                            ("case_num", "n_component"),
                            gp_models[goal - 1]
                            .predict(
                                dataset=reanalysis_df_for_goal.fillna(0),
                            )
                            .values,
                        )
                    },
                    coords={"case_num": reanalysis_df_for_goal.index},
                )
            )
            prediction_ds_in_buoys = xr.Dataset(
                {
                    "hs": (
                        ("time", "stations"),
                        prediction_ds_in_points.point_hm0.values,
                    ),
                    "tp": (
                        ("time", "stations"),
                        prediction_ds_in_points.point_tp.values,
                    ),
                    "dp": (
                        ("time", "stations"),
                        get_degrees_from_uv(
                            prediction_ds_in_points.point_wavdir_u.values,
                            prediction_ds_in_points.point_wavdir_v.values,
                        ),
                    ),
                    "spr": (
                        ("time", "stations"),
                        reanalysis_df_for_goal.ds.repeat(177).values.reshape(744, 177),
                    ),
                },
                coords={
                    "time": reanalysis_df_for_goal.index,
                },
            )
            prediction_ds_in_buoys["hs"] = (
                prediction_ds_in_buoys.hs * reanalysis_ds_for_goal.hs
            )
            prediction_ds_in_buoys = (
                prediction_ds_in_buoys.expand_dims(
                    {"goal": [goal], "partition": [partition]}
                )
                .isel(stations=slice(-9, None))
                .load()
                .drop(["seapoint", "latitude", "longitude"])
            )

            # Save the prediction in buoys
            reconstruction_for_each_goal.append(prediction_ds_in_buoys)

ValueError: cannot rename 'hs' because it is not a variable or dimension in this dataset

In [ ]:
len(reconstruction_for_each_goal), reconstruction_for_each_goal[0]

(50,
 <xarray.Dataset> Size: 220kB
 Dimensions:    (goal: 1, partition: 1, time: 744, stations: 9)
 Coordinates:
   * goal       (goal) int64 8B 1
   * partition  (partition) object 8B 'bulk'
   * time       (time) datetime64[ns] 6kB 2011-12-01 ... 2011-12-31T23:00:00
 Dimensions without coordinates: stations
 Data variables:
     hs         (goal, partition, time, stations) float64 54kB -0.05102 ... -0...
     tp         (goal, partition, time, stations) float64 54kB 7.624 ... 5.472
     dp         (goal, partition, time, stations) float64 54kB 296.4 ... 267.8
     spr        (goal, partition, time, stations) float64 54kB 47.1 47.1 ... 47.6)

In [ ]:
datasesito = xr.merge(reconstruction_for_each_goal)
datasesito

<xarray.Dataset> Size: 11MB
Dimensions:    (time: 744, goal: 10, partition: 5, stations: 9)
Coordinates:
  * time       (time) datetime64[ns] 6kB 2011-12-01 ... 2011-12-31T23:00:00
  * goal       (goal) int64 80B 1 2 3 4 5 6 7 8 9 10
  * partition  (partition) object 40B 'bulk' 'sea' 'swell1' 'swell2' 'swell3'
Dimensions without coordinates: stations
Data variables:
    hs         (goal, partition, time, stations) float64 3MB -0.05102 ... nan
    tp         (goal, partition, time, stations) float64 3MB 7.624 ... 0.0996
    dp         (goal, partition, time, stations) float64 3MB 296.4 ... 6.887
    spr        (goal, partition, time, stations) float64 3MB 47.1 47.1 ... nan

In [ ]:
partitions = (
    datasesito.isel(partition=[1, 2, 3, 4])
    .rename({"stations": "site"})
    .stack(part=("goal", "partition"))
)
partitions

<xarray.Dataset> Size: 9MB
Dimensions:    (time: 744, site: 9, part: 40)
Coordinates:
  * time       (time) datetime64[ns] 6kB 2011-12-01 ... 2011-12-31T23:00:00
  * part       (part) object 320B MultiIndex
  * goal       (part) int64 320B 1 1 1 1 2 2 2 2 3 3 ... 8 8 9 9 9 9 10 10 10 10
  * partition  (part) object 320B 'sea' 'swell1' 'swell2' ... 'swell2' 'swell3'
Dimensions without coordinates: site
Data variables:
    hs         (time, site, part) float64 2MB -0.04887 -0.002959 nan ... nan nan
    tp         (time, site, part) float64 2MB 7.653 16.34 ... 0.09961 0.0996
    dp         (time, site, part) float64 2MB 297.2 352.9 46.02 ... 6.888 6.887
    spr        (time, site, part) float64 2MB 47.0 10.5 nan nan ... 12.3 nan nan

In [ ]:
from wavespectra.construct import construct_partition

dir = np.arange(0, 360, 5)
freq = np.logspace(np.log10(0.035), np.log10(0.5), 29)
full_spec = construct_partition(
    freq_name="jonswap",
    freq_kwargs={
        "freq": freq,
        "fp": 1 / partitions.tp,
        "hs": partitions.hs,
    },
    dir_name="cartwright",
    dir_kwargs={
        "dir": dir,
        "dm": partitions.dp,
        "dspr": partitions.spr,
    },
)
full_spec

<xarray.DataArray (time: 744, site: 9, part: 40, freq: 29, dir: 72)> Size: 4GB
array([[[[[3.71017633e-108, 3.32253624e-108, 2.94310713e-108, ...,
           4.86565795e-108, 4.48770397e-108, 4.10051826e-108],
          [4.95844885e-075, 4.44038895e-075, 3.93330260e-075, ...,
           6.50268719e-075, 5.99757225e-075, 5.48011961e-075],
          [1.93248992e-052, 1.73058292e-052, 1.53295272e-052, ...,
           2.53433641e-052, 2.33747454e-052, 2.13580421e-052],
          ...,
          [4.58569835e-008, 4.10658351e-008, 3.63761728e-008, ...,
           6.01384885e-008, 5.54670584e-008, 5.06815261e-008],
          [2.86341241e-008, 2.56424241e-008, 2.27140943e-008, ...,
           3.75518146e-008, 3.46348694e-008, 3.16466762e-008],
          [1.78575371e-008, 1.59917775e-008, 1.41655383e-008, ...,
           2.34190129e-008, 2.15998738e-008, 1.97363011e-008]],

         [[1.23254911e-010, 8.04001152e-011, 4.18572949e-011, ...,
           1.16164944e-010, 1.48102048e-010, 1.51052026e-010],
          [3.06706716e-009, 2.00067121e-009, 1.04157419e-009, ...,
           2.89064089e-009, 3.68536169e-009, 3.75876875e-009],
          [2.37841375e-008, 1.55145736e-008, 8.07707900e-009, ...,
           2.24160074e-008, 2.85788163e-008, 2.91480649e-008],
...
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000]],

         [[0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          ...,
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000],
          [0.00000000e+000, 0.00000000e+000, 0.00000000e+000, ...,
           0.00000000e+000, 0.00000000e+000, 0.00000000e+000]]]]])
Coordinates:
  * time       (time) datetime64[ns] 6kB 2011-12-01 ... 2011-12-31T23:00:00
  * part       (part) object 320B MultiIndex
  * freq       (freq) float64 232B 0.035 0.03849 0.04232 ... 0.4135 0.4547 0.5
  * dir        (dir) int64 576B 0 5 10 15 20 25 30 ... 330 335 340 345 350 355
  * goal       (part) int64 320B 1 1 1 1 2 2 2 2 3 3 ... 8 8 9 9 9 9 10 10 10 10
  * partition  (part) object 320B 'sea' 'swell1' 'swell2' ... 'swell2' 'swell3'
Dimensions without coordinates: site
Attributes:
    globwave_name:          significant_wave_height
    long_name:              Significant height of wind and swell waves
    standard_name:          sea_surface_wave_significant_height
    units:                  m
    valid_max:              32000
    valid_min:              0
    coverage_content_type:  modelResult
    mesh:                   SMC

In [ ]:
full_spec.sum(dim="part").to_netcdf("outputs/201112_500_spectra_ERA5.nc")